# Evolution of Modern Language-Model Architectures

> MiniGPT already has the basic shape of a modern language model. Scaling from a few layers to dozens and from thousands of Tokens to million-Token contexts exposes unstable training, insufficient memory, and excessive computation.
>
> The Block input and output shapes stay fixed, while internal components evolve around six questions: how information crosses deep networks, how position is represented, how history is cached, how long contexts are read, how FFN capacity grows, and how new structures receive stable training signals.
>
> Every experiment begins with the same teaching Block, changes one component, and compares its computation, benefits, and costs.

The 2017 Transformer used Encoder and Decoder stacks, Post-Norm, ReLU FFNs, sinusoidal positions, and MHA. GPT-2 moved to Decoder-only, Pre-Norm, GELU, and learned absolute positions. LLaMA adopted RMSNorm, RoPE, SwiGLU, and later GQA. DeepSeek-V3 added MLA, MoE, and MTP. Later models added QK-Norm, QK-Clip, Muon, sparse retrieval, and hybrids of recurrent state with periodic global Attention. These systems do not discard the Transformer; they select different component combinations for different constraints.


## The Basic Transformer Block

Let's review the teaching Block on which every later modification builds.

`self.attention(x, x, x)` passes Query, Key, and Value:

- **Query**: what the current position seeks;
- **Key**: what clues each position offers;
- **Value**: the information retrieved and mixed.

In Self-Attention, all three are separate projections of the same sequence `x`. Cross-Attention may instead use decoder state as Query and encoder output as Key and Value.


In [ ]:
# === This is the teaching-version MiniGPT Block - our "upgrade target" ===

import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

class FeedForward_Old(nn.Module):
    """Original FFN: two Linear layers with ReLU in between"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class TransformerBlock_Old(nn.Module):
    """Post-Norm + LayerNorm + ReLU FFN (teaching version)"""
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = FeedForward_Old(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)  # ordinary LayerNorm
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # Self-Attention: Q, K, V all come from the same x
        # mask blocks future tokens, preventing the Decoder from peeking at the answer
        attn_out, _ = self.attention(
            x, x, x,
            attn_mask=mask,
            need_weights=False,
        )

        # Post-Norm: sublayer first, then +residual, finally Norm
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x

print("\✅ This is the teaching-version MiniGPT Block. Next we upgrade it component by component.")
print("Keep the input and output shapes unchanged while adjusting information flow, the FFN, positions, KV storage, context access, and training methods.")


## 1. Normalization and Residual Connections in Deep Networks

With only two Blocks, Post-Norm and Pre-Norm appear similar. Across dozens of layers, however, gradients in Post-Norm repeatedly pass through normalization derivatives and early layers receive weaker signals. We first compare their gradient paths, then calculate LayerNorm and RMSNorm by hand. Placement and formula solve related but distinct problems.


**Post-Norm**

In the teaching-version Block:

```python
x = x + attention(x)   # residual: add input and Attention output
x = norm(x)            # LayerNorm: standardize
```

LayerNorm comes **after** Attention, hence the name Post-Norm.

The full computation graph:

```
  x --> Attention --> + --> LayerNorm --> output
  |                  ^
  +------------------+  (residual connection)
```

The residual connection adds to the Attention output, and then both pass through LayerNorm together. That is, **the residual path goes through LayerNorm**.

The role of the residual connection is to give gradients a direct backpropagation path that is not attenuated by Attention and the FFN.

But in Post-Norm, a LayerNorm is inserted into this path. LayerNorm scales gradients, so after passing through it the gradient magnitude may be altered.

In a 4-layer network this isn't a problem. But in a 40-layer or 80-layer network, every layer passes through a LayerNorm, and the scaling effect on gradients accumulates - they may explode or vanish.

**Pre-Norm**

Pre-Norm moves LayerNorm to **before** Attention:

```python
x = x + attention(norm(x))   # Norm first, then Attention
```

Computation graph:

```
  x --> LayerNorm --> Attention --> + --> output
  |                                 ^
  +---------------------------------+  (residual connection, doesn't go through Norm)
```

The key change is that the residual connection bypasses LayerNorm; gradients on the residual path are not scaled.

Let's compare with two minimal networks:


In [ ]:
# === Post-Norm vs Pre-Norm: seeing gradient flow through hand calculation ===
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== Post-Norm vs Pre-Norm: gradient flow comparison ===")
print()

# Build two simplified deep networks (using FFN instead of Attention, focusing on Norm placement)
d_model = 4
num_layers = 8  # start with 8 layers to see the effect

class Deep_PostNorm(nn.Module):
    """Post-Norm: x = Norm(x + FFN(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = norm(x + F.relu(layer(x)))
        return x

class Deep_PreNorm(nn.Module):
    """Pre-Norm: x = x + FFN(Norm(x))"""
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer, norm in zip(self.layers, self.norms):
            x = x + F.relu(layer(norm(x)))
        return x

# Same initialization
torch.manual_seed(42)
model_post = Deep_PostNorm(d_model, num_layers)
torch.manual_seed(42)
model_pre = Deep_PreNorm(d_model, num_layers)

# Random input
x = torch.randn(2, d_model)
target = torch.randn(2, d_model)

# Forward + Backward
loss_post = F.mse_loss(model_post(x), target)
loss_post.backward()

loss_pre = F.mse_loss(model_pre(x), target)
loss_pre.backward()

# Look at gradient norm for each layer
print(f"Network depth: {num_layers} layers")
print()
print(f"{'Layer':>6s}  {'Post-Norm grad':>16s}  {'Pre-Norm grad':>16s}")
print("-" * 42)

grad_post_values = []
grad_pre_values = []
for i in range(num_layers):
    grad_post = model_post.layers[i].weight.grad.norm().item()
    grad_pre = model_pre.layers[i].weight.grad.norm().item()
    grad_post_values.append(grad_post)
    grad_pre_values.append(grad_pre)
    print(f"Layer {i+1}:  {grad_post:>16.6f}  {grad_pre:>16.6f}")

# Most important: look at layer 1 gradient (bottom layer)
grad_post_first = model_post.layers[0].weight.grad.norm().item()
grad_pre_first = model_pre.layers[0].weight.grad.norm().item()

print()
print(f"Most critical - bottom layer (Layer 1) gradient comparison:")
print(f"  Post-Norm Layer 1 gradient: {grad_post_first:.6f}")
print(f"  Pre-Norm  Layer 1 gradient: {grad_pre_first:.6f}")
print(f"  Pre-Norm / Post-Norm = {grad_pre_first/grad_post_first:.2f}x")
print()
print("Key observation: the two structures distribute gradients differently; Pre-Norm preserves a residual route that bypasses normalization.")
print("This small experiment uses a fixed initialization; it does not imply that every model has the same ratio.")

layers = list(range(1, num_layers + 1))
plt.figure(figsize=(8, 4))
plt.plot(layers, grad_post_values, marker='o', label='Post-Norm')
plt.plot(layers, grad_pre_values, marker='o', label='Pre-Norm')
plt.yscale('log')
plt.xlabel('Layer index (input to output)')
plt.ylabel('Weight gradient norm (log scale)')
plt.title('Gradient distribution across an 8-layer toy network')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


**Post-Norm vs Pre-Norm comparison**

- **Post-Norm**: sublayer computation first, then normalization -> the residual path goes through Norm, and gradients can be truncated in deep networks
- **Pre-Norm**: normalize the residual path first, then do the sublayer computation -> the residual path stays open, deep training is more stable

In deep networks, Pre-Norm offers higher training stability - you can use larger learning rates and fewer warmup steps.


**Post-Norm and Pre-Norm Computation Graphs**

```text
Old — Post-Norm                         New — Pre-Norm
(sublayer → residual → Norm)            (Norm → sublayer → residual)

x → Attention → + → LayerNorm           x → LayerNorm → Attention → +
|               ↑                       |                           ↑
+---------------+                       +---------------------------+
residual passes through Norm            residual bypasses Norm
```

| | Post-Norm | Pre-Norm |
|:---|:---|:---|
| Norm position | after sublayer + residual | before sublayer |
| Residual path | scaled by Norm | direct identity path |
| Backpropagation | repeatedly crosses Norm | includes a direct gradient path |
| Deep training | sensitive to initialization and warmup | usually easier to stabilize |
| Examples | original Transformer | GPT-2, LLaMA, many Decoder-only models |


**LayerNorm hand calculation**

Suppose a token's vector is `[1, 3, 5, 7]` (4 dimensions, chosen for easy hand calculation; here we also use 4 dimensions).

What LayerNorm does is turn this vector's mean into 0 and its standard deviation into 1. You can think of it as standardizing a set of scores: regardless of the raw values, after processing the mean is 0 and the spread is uniform.

The variance computation is where it is easiest to get stuck. Its meaning is how spread out this set of numbers is from the average.

First, look at how far each number is from the average:

```text
mean mu = 4

1 differs from 4 by -3
3 differs from 4 by -1
5 differs from 4 by  1
7 differs from 4 by  3
```

If we directly sum these differences:

```text
(-3) + (-1) + 1 + 3 = 0
```

`[1, 3, 5, 7]` is clearly spread out, but positive and negative numbers cancel each other, making it look as if there is no spread at all. So we cannot simply average the differences.

The solution is to square each difference:

```text
(-3)^2 = 9
(-1)^2 = 1
 1^2  = 1
 3^2  = 9
```

Squaring does two things: negatives become positives and no longer cancel; the farther from the mean, the larger the penalty - e.g., a difference of 3 becomes 9.

Finally, take the average:

```text
variance sigma^2 = (9 + 1 + 1 + 9) / 4 = 5
```

The variance means, on average, how large the squared deviation of each number from the mean is.

Concrete steps:
```
1. Compute the mean   mu = (x1 + x2 + x3 + x4) / 4
2. Compute each number's deviation from the mean: x1-mu, x2-mu, x3-mu, x4-mu
3. Square the deviations to avoid positive/negative cancellation: (x1-mu)^2, (x2-mu)^2, ...
4. Average the squared deviations to get the variance sigma^2
5. Compute the standard deviation sigma = sqrt(sigma^2)
6. Normalize   x' = (x - mu) / sigma
7. Scale       y = gamma * x' + beta
```

Here we divide by 4 because we are averaging over the 4 dimensions of this token. PyTorch's `LayerNorm` likewise treats all dimensions of the current vector as a group of numbers and computes their own mean and variance.

gamma and beta are learnable parameters, letting the model decide for itself "how much to scale" and "which way to shift" after normalization.

Now let's hand-calculate with `[1, 3, 5, 7]`:


In [ ]:
# === LayerNorm hand calculation ===
import torch

print("=== LayerNorm hand calculation: input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Step 1: mean
mu = x.mean()
print(f"Step 1 - mean mu = (1+3+5+7)/4 = {mu:.1f}")

# Step 2: variance (dividing by N, not N-1)
var = torch.mean((x - mu) ** 2)
print(f"Step 2 - variance sigma^2 = ((1-4)^2+(3-4)^2+(5-4)^2+(7-4)^2)/4")
print(f"        = (9 + 1 + 1 + 9)/4 = {var:.1f}")

# Step 3: standard deviation
sigma = torch.sqrt(var)
print(f"Step 3 - std sigma = sqrt({var:.1f}) = {sigma:.4f}")

# Step 4: normalize
x_norm = (x - mu) / sigma
print(f"Step 4 - normalize: (x - 4)/{sigma:.4f}")
for i, (xi, xni) in enumerate(zip(x.tolist(), x_norm.tolist())):
    print(f"         x[{i}] = ({xi:.1f} - 4) / {sigma:.4f} = {xni: .4f}")

print(f"         after normalization: {[f'{v:.4f}' for v in x_norm.tolist()]}")
print(f"         mean: {x_norm.mean():.4f} (=0), std: {x_norm.std(unbiased=False):.4f} (=1)")

# Step 5: scale (assuming gamma=[1,1,1,1], beta=[0,0,0,0])
# Initially gamma is all 1s and beta is all 0s, so the output equals x_norm
print(f"Step 5 - scale: when gamma=1, beta=0, output = normalized result")
print(f"         gamma and beta are learned during training, letting the model decide how to adjust")


**Computational Cost of LayerNorm**

LayerNorm computes a mean and variance at every Token in every layer. Variance depends on the mean, requiring another pass through the dimensions. In large models this repeated work matters.

```text
LayerNorm(x) = (x - mean) / std * gamma + beta
                re-center      re-scale
```

RMSNorm separates these properties and retains only rescaling. Experiments in Zhang & Sennrich (2019) found similar training stability without explicit recentering. This is empirical, not a consequence of linear-layer bias; many modern projections omit bias.

```text
RMS(x) = sqrt(mean(x^2))
RMSNorm(x) = x / RMS(x) * gamma
```

RMSNorm avoids the mean and centered squared deviations and commonly omits beta. LLaMA, Qwen, Mistral, and DeepSeek use it. The following cell calculates RMSNorm for `[1, 3, 5, 7]`.


In [ ]:
# === RMSNorm hand calculation ===
import torch
import torch.nn as nn

print("=== RMSNorm hand calculation: input x = [1, 3, 5, 7] ===")
print()

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# The only step: compute RMS
mean_sq = torch.mean(x ** 2)
rms = torch.sqrt(mean_sq)

print(f"Step 1 - mean of squares = (1^2+3^2+5^2+7^2)/4")
print(f"          = (1+9+25+49)/4")
print(f"          = {84/4:.1f}")
print(f"    RMS = √{mean_sq:.1f} = {rms:.4f}")
print()

# RMSNorm output
x_rmsnorm = x / rms
print(f"Step 2 — RMSNorm = x / {rms:.4f}")
for i, (xi, xri) in enumerate(zip(x.tolist(), x_rmsnorm.tolist())):
    print(f"         x[{i}] = {xi:.1f} / {rms:.4f} = {xri:.4f}")

print(f"\nRMSNorm result: {[f'{v:.4f}' for v in x_rmsnorm.tolist()]}")

# Verify: RMS after RMSNorm should be 1
rms_after = torch.sqrt(torch.mean(x_rmsnorm ** 2))
print(f"\nVerification - RMS after normalization = {rms_after:.4f} (should = 1) \✅")
print(f"        Note: mean after RMSNorm != 0 (mean = {x_rmsnorm.mean():.4f})")
print(f"        This is the difference from LayerNorm - no mean subtraction!")
print()

# Comparison
ln = nn.LayerNorm(4, elementwise_affine=False)  # disable gamma, beta for fair comparison
x_ln = ln(x)
ln_values = [f"{v:.4f}" for v in x_ln.tolist()]
print(f"LayerNorm result: {ln_values}")
print(f"  mean={x_ln.mean():.4f}, std={x_ln.std(unbiased=False):.4f}")
rms_values = [f"{v:.4f}" for v in x_rmsnorm.tolist()]
print(f"RMSNorm  result: {rms_values}")
print(f"  mean={x_rmsnorm.mean():.4f}, RMS={rms_after:.4f}")
print(f"\n-> Different values, but both achieve 'standardization'")
print(f"-> LayerNorm forces mean=0, RMSNorm doesn't (but in practice it's often good enough)")


In [ ]:
# === RMSNorm full implementation ===

import torch
import torch.nn as nn

torch.manual_seed(42)

class RMSNorm(nn.Module):
    """
    RMSNorm: only scaling, no centering

    Formula: y = x / RMS(x) * gamma
    RMS(x) = sqrt(mean(x^2) + eps)

    - No beta (bias), because there's no mean subtraction to compensate for
    - eps prevents division by zero (e.g., when input is all zeros)
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))  # only gamma

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        # Compute RMS along the last dimension
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma

# Test
d_model = 8
rmsn = RMSNorm(d_model)
ln = nn.LayerNorm(d_model)

batch = torch.randn(2, 4, d_model) * 3  # simulate 2 batches, 4 tokens, 8 dims
out_rms = rmsn(batch)
out_ln = ln(batch)

print("=== RMSNorm implementation verification ===")
print(f"Input shape: {batch.shape}")
print(f"RMSNorm output shape: {out_rms.shape}")
print(f"Output RMS (should be ~= 1): {torch.sqrt(torch.mean(out_rms**2, dim=-1))[0].tolist()}")
print()

# Parameter count comparison
ln_params = sum(p.numel() for p in ln.parameters())
rmsn_params = sum(p.numel() for p in rmsn.parameters())
print(f"LayerNorm params: gamma({d_model}) + beta({d_model}) = {ln_params}")
print(f"RMSNorm  params: gamma({d_model}) = {rmsn_params}")
print(f"-> Parameters halved, but that's not the main point - the main point is computing one fewer statistic")


In [ ]:
# === Before and after: run the same input through LayerNorm and RMSNorm ===
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

x = torch.tensor([1.0, 3.0, 5.0, 7.0])

# Old: LayerNorm = (x - mu) / sigma * gamma + beta
mu = x.mean()
sigma = x.std(unbiased=False)
ln_out = (x - mu) / sigma

# New: RMSNorm = x / RMS(x) * gamma
rms = torch.sqrt(torch.mean(x ** 2))
rmsn_out = x / rms

print("=== RMSNorm hand calculation: input x = [1, 3, 5, 7] ===")
print()
print(f"{'Step':<20s} {'LayerNorm (old)':>16s} {'RMSNorm (new)':>16s}")
print("-" * 56)
print(f"{'Compute mean?':<20s} {'yes, mean=4.0':>16s} {'no, skip it':>16s}")
print(f"{'Variance/mean square?':<20s} {'var=5.0':>16s} {'ms=21.0':>16s}")
print(f"{'Square root':<20s} {'std=2.236':>16s} {'RMS=4.583':>16s}")
print(f"{'Subtract mean?':<20s} {'yes':>16s} {'no':>16s}")
print(f"{'Divide by statistic':<20s} {'(x-4)/2.236':>16s} {'x/4.583':>16s}")
print(f"{'Passes over data':<20s} {'2 passes':>16s} {'1 pass':>16s}")
print(f"{'Learnable parameters':<20s} {'gamma + beta':>16s} {'gamma only':>16s}")
print()
print(f"LayerNorm output: {[f'{v:.4f}' for v in ln_out.tolist()]}")
print(f"RMSNorm output:  {[f'{v:.4f}' for v in rmsn_out.tolist()]}")
print(f"  LayerNorm mean = {ln_out.mean():.4f} (=0)")
print(f"  RMSNorm mean = {rmsn_out.mean():.4f} (not zero, but training still works)")

# Visualize the effects of the two normalization methods
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, vals, title, color in [
    (axes[0], ln_out, 'LayerNorm (old)', '#dc2626'),
    (axes[1], rmsn_out, 'RMSNorm (new)', '#2563eb'),
]:
    ax.bar(range(len(vals)), vals.tolist(), color=color, width=0.5)
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([f'x[{i}]' for i in range(len(vals))])
    ax.set_title(title)
    ax.set_ylabel('Normalized value')
    ax.grid(True, axis='y', alpha=0.3)
plt.suptitle('Same input [1,3,5,7], two normalization methods', fontsize=11)
plt.tight_layout()
plt.show()


**Sandwich Norm**

Gemma 3 adds normalization after each sublayer as well as before it:

```text
Pre-Norm:       x → RMSNorm → Attention → residual → RMSNorm → FFN → residual
Sandwich Norm:  x → RMSNorm → Attention → RMSNorm → residual
                   → RMSNorm → FFN → RMSNorm → residual
```

| | Standard Pre-Norm | Sandwich Norm |
|:---|:---|:---|
| Norm count | 2 | 4 |
| Sublayer output | unnormalized | normalized again |
| Goal | gradient stability | also bound residual-branch activations |
| Examples | LLaMA, Qwen3, DeepSeek | Gemma 3 |

The extra controls activation drift but costs two additional normalizations per Block.


**Extensions to Residual Connections**

| Era | Method | Residual Path | Main Idea |
|:---|:---|:---|:---|
| 2017 | Post-Norm | through Norm | difficult deep optimization |
| 2019+ | Pre-Norm | bypasses Norm | stable direct path |
| 2022 | DeepNorm | scaled then normalized | coefficients designed for very deep Transformers |
| 2026 | mHC | multiple mixed streams | constrained learned mixing |

DeepNorm scales the identity and sublayer branches with paired coefficients. DeepSeek V4's manifold-constrained hyper-connections carry several streams and learn how to mix them under numerical constraints. The standard form $x_{l+1}=x_l+f_l(x_l)$ has a fixed identity/residual ratio; multi-stream residuals learn how much information to preserve or transform at each layer.


## 2. Feed-Forward Networks and Gating

Attention decides which Tokens to read; the FFN decides how each Token transforms the information it has read. Its nonlinear computation often contains a large fraction of Block parameters, with `d_ff` several times `d_model`. Replacing ReLU with SwiGLU therefore upgrades the main per-Token computation rather than a minor activation.

```text
FFN(x) = W2 · ReLU(W1 · x)
ReLU: negative → 0, positive → unchanged
```

The next cell performs a concrete hand calculation.


In [ ]:
# === ReLU hand calculation ===
import torch
import torch.nn.functional as F

print("=== What ReLU does to each number ===")
print()

# Simulate output of W1 . x (extended to 8 dimensions)
hidden = torch.tensor([0.5, -2.0, 3.0, -0.1, 0.0, -5.0, 1.5, -0.01])

print(f"Input (result of W1 . x): {[f'{v:.2f}' for v in hidden.tolist()]}")
print()

relu_out = F.relu(hidden)
print("What ReLU does:")
for i, (h, r) in enumerate(zip(hidden.tolist(), relu_out.tolist())):
    action = "keep" if h >= 0 else "-> 0 (kill)"
    print(f"  Position {i}: {h:6.2f} -> {r:6.2f}  {action}")

print(f"\nProblem: {sum(1 for h in hidden if h < 0)} out of 8 positions are killed outright")
print(f"      Among them, -0.1 and -0.01 are only 'slightly negative', yet they're killed too")
print(f"      -> ReLU is too harsh: all negatives are discarded completely")


**The SwiGLU gating mechanism**

ReLU directly clamps negatives to 0: if a value is partly useful and partly useless, the entire value disappears.

The idea of gating is to use two parallel channels:
- one does a linear transform to produce "information"
- another does a linear transform + Swish to produce a "weight" between 0 and 1
- the two are multiplied element-wise, letting the weight decide how much each component passes through

The effect: if some component is partly useful, the gate gives a weight of 0.3 and it still partially passes through, instead of vanishing entirely.

From a matrix-operation perspective, the change from two matrices to three matrices:

```
Teaching-version FFN:  x -> W1 -> ReLU   -> W2 -> output
SwiGLU FFN:  x -> W_up   |
              x -> W_gate +-> multiply -> W_down -> output
```

Formula:

```
SwiGLU(x) = (W_up . x)  (*)  Swish(W_gate . x)
              ^ information channel    ^ gating channel

Swish(a) = a * sigmoid(a) = a / (1 + e^(-a))
```

Swish is a smooth activation function that, unlike ReLU, doesn't hard-clip to 0 in the negative region.


In [ ]:
# === ReLU vs Swish hand calculation comparison ===
import math

print("=== Activation function comparison: ReLU vs Swish ===")
print()

x_vals = [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]

print(f"{'x':>8s}  {'ReLU(x)':>10s}  {'Swish(x)':>10s}  {"'Note'"}")
print("-" * 52)

for x in x_vals:
    relu = max(0, x)
    swish = x / (1 + math.exp(-x))  # x * sigmoid(x)
    note = ""
    if x < 0:
        note = f"ReLU kills, Swish keeps {swish:.4f}"
    elif x == 0:
        note = "ReLU stuck at 0, Swish is smooth"
    else:
        note = "both pass"
    print(f"{x:>8.1f}  {relu:>10.1f}  {swish:>10.4f}  {note}")

print()
print("Key observation:")
print("  1. Positive region: Swish ~= ReLU (slightly smaller), similar behavior")
print("  2. Negative region: ReLU = 0 (completely killed), Swish keeps a small negative value")
print("  3. At x=0: ReLU is not differentiable (mathematically), Swish is smooth and differentiable")
print()
print("Why are 'small negative values' important?")
print("  Gradient = d(loss)/dx, and the Swish derivative remains nonzero for negative inputs")
print("  -> Even if this neuron is currently 'not very active', gradients can still flow through")
print("  -> ReLU neurons may 'stop learning for a long time' (once in the negative region, gradient=0, can't come back)")


In [ ]:
# === Hand-calculating one step of SwiGLU ===
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=== SwiGLU hand calculation ===")
print()

# Tiny example: d_model=4, simulating one piece of information
d_model = 4
x = torch.tensor([[1.0, -0.5, 2.0, 0.3]])  # one token

# Simplified versions of three weight matrices (hand-constructed for easy observation)
# In practice these would be nn.Linear weights
torch.manual_seed(1)
W_up = nn.Linear(d_model, d_model, bias=False)
W_gate = nn.Linear(d_model, d_model, bias=False)
W_down = nn.Linear(d_model, d_model, bias=False)

# Information channel
up = W_up(x)
# Gating channel: linear transform then Swish
gate = F.silu(W_gate(x))  # silu = Swish
# Gated result = information * gate
gated = up * gate
# Output projection
out = W_down(gated)

print(f"Input x: {x.tolist()}")
print()
print(f"Information channel (W_up . x):     {[f'{v:.3f}' for v in up[0].tolist()]}")
print(f"Gating channel Swish(W_gate . x): {[f'{v:.3f}' for v in gate[0].tolist()]}")
print(f"After gating (up * gate):     {[f'{v:.3f}' for v in gated[0].tolist()]}")
print(f"Output (W_down . gated):    {[f'{v:.3f}' for v in out[0].tolist()]}")
print()
print("When the gate is near zero, it suppresses the corresponding information component.")
print("The gate is not restricted to 0 through 1; it can also change a component's magnitude and sign.")


In [ ]:
# === Complete SwiGLU FFN implementation ===

import torch.nn as nn
import torch.nn.functional as F

class FeedForward_SwiGLU(nn.Module):
    """
    SwiGLU FFN, a structure commonly used by LLaMA-style models.

    Formula: FFN(x) = W_down * (Swish(W_gate*x) elementwise-multiplied by W_up*x)

    Three weight matrices:
    - W_gate: gate projection from d_model to d_ff
    - W_up: information projection from d_model to d_ff
    - W_down: output projection from d_ff to d_model

    The original FFN has two matrices, while SwiGLU has three. To keep the
    parameter count comparable, reduce d_ff from 4d to approximately 8d/3.
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # 3 * d_model * d_ff is approximately 2 * d_model * 4d_model
            # Therefore d_ff is approximately 8/3 * d_model
            d_ff = int(8 / 3 * d_model)
            d_ff = ((d_ff + 255) // 256) * 256  # Round up to a multiple of 256
            d_ff = max(d_ff, d_model)  # Keep it at least as large as d_model

        self.W_gate = nn.Linear(d_model, d_ff, bias=False)
        self.W_up = nn.Linear(d_model, d_ff, bias=False)
        self.W_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.W_down(F.silu(self.W_gate(x)) * self.W_up(x))

# Compare parameter counts
d_model = 512
ffn_old = FeedForward_Old(d_model)
ffn_new = FeedForward_SwiGLU(d_model)

old_p = sum(p.numel() for p in ffn_old.parameters())
new_p = sum(p.numel() for p in ffn_new.parameters())

print("=== SwiGLU versus ReLU FFN parameter counts ===")
print(f"d_model = {d_model}")
print()
print(f"ReLU FFN:  W1({d_model}x{4*d_model}) + W2({4*d_model}x{d_model})")
print(f"           = {d_model*4*d_model:,} + {4*d_model*d_model:,}")
print(f"           = {old_p:,} parameters")
print()
d_ff_s = ((int(8/3*d_model) + 255) // 256) * 256
print("SwiGLU FFN:")
print(f"  W_gate({d_model}x{d_ff_s}) + W_up({d_model}x{d_ff_s})")
print(f"  + W_down({d_ff_s}x{d_model})")
print(f"           = {d_model*d_ff_s:,} + {d_model*d_ff_s:,} + {d_ff_s*d_model:,}")
print(f"           = {new_p:,} parameters")
print()
print(f"Parameter ratio: {new_p/old_p:.2f}x, where about 1 is the target")
print("Key observation: after reducing d_ff, SwiGLU adds a gate branch while keeping the total parameter count close to the old FFN.")

plt.figure(figsize=(6, 3.8))
bars = plt.bar(['ReLU FFN', 'SwiGLU FFN'], [old_p, new_p],
               color=['#dc2626', '#2563eb'])
plt.ylabel('Parameter count')
plt.title('Parameter-matched FFN comparison')
plt.grid(True, axis='y', alpha=0.3)
for bar, value in zip(bars, [old_p, new_p]):
    plt.text(bar.get_x() + bar.get_width() / 2, value, f'{value / 1e6:.2f}M',
             ha='center', va='bottom')
plt.tight_layout()
plt.show()


In [ ]:
# === Before and after: ReLU versus Swish curves and negative-input behavior ===
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-4, 4, 200)
relu = np.maximum(0, x)
swish = x / (1 + np.exp(-x))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: activation curves
axes[0].plot(x, relu, color='#dc2626', linewidth=2.5, label='ReLU (old)')
axes[0].plot(x, swish, color='#2563eb', linewidth=2.5, label='Swish (new)')
axes[0].axhline(0, color='gray', linewidth=0.8)
axes[0].axvline(0, color='gray', linewidth=0.8)
axes[0].fill_between(x[x < 0], 0, relu[x < 0], alpha=0.15, color='#dc2626',
                     label='ReLU kills negatives')
axes[0].set_xlabel('input')
axes[0].set_ylabel('output')
axes[0].set_title('Activation function: ReLU vs Swish')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: gradients
d_relu = (x > 0).astype(float)
d_swish = 1 / (1 + np.exp(-x)) + x * np.exp(-x) / (1 + np.exp(-x)) ** 2
axes[1].plot(x, d_relu, color='#dc2626', linewidth=2.5, label="ReLU grad (old)")
axes[1].plot(x, d_swish, color='#2563eb', linewidth=2.5, label="Swish grad (new)")
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].axvline(0, color='gray', linewidth=0.8)
axes[1].fill_between(x[x < 0], 0, d_relu[x < 0], alpha=0.15, color='#dc2626',
                     label='ReLU grad = 0 in negatives')
axes[1].set_xlabel('input')
axes[1].set_ylabel('gradient')
axes[1].set_title('Gradient: ReLU vs Swish in negative region')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  Old (ReLU):  output = 0 and gradient = 0 for negative inputs, so a negative neuron can stop learning")
print("  New (Swish): output and gradient remain nonzero for negative inputs, so the neuron can keep learning")
print("  SwiGLU = Swish gate branch * linear information branch; the gate can suppress or amplify information.")


**Dense FFN versus MoE**

| | Dense FFN | Fine-Grained MoE |
|:---|:---|:---|
| Structure | one FFN shared by every Token | $N$ experts + router; select $K$ |
| Parameters vs compute | coupled | decoupled: many total parameters, few active |
| Typical setup | `d_ff = 4 × d_model` | 128 experts, 8 active |
| Examples | GPT-2, LLaMA, dense Qwen3 | DeepSeek-V3, Kimi K2, GLM-4.5, Llama 4 |

The MoE chapter implements routing, top-k selection, balancing, and auxiliary-loss-free routing. Here we focus on the architectural contrast and the change from Softmax to sigmoid gates.


**Gating: Softmax → Sigmoid**

| | Softmax Gate | Sigmoid Gate |
|:---|:---|:---|
| Formula | $g_i=e^{z_i}/\sum_j e^{z_j}$ | $g_i=\sigma(z_i)$ |
| Normalization | sums to 1 | independent values |
| After top-k | renormalize selected experts | use selected sigmoid values directly |
| Behavior | amplifies competition | smoother independent gates |
| Examples | Mixtral, GShard | DeepSeek-V3, GLM-4.5 |

```python
# Softmax gate
topk_logits, topk_idx = router(x).topk(k)
weights = F.softmax(topk_logits, dim=-1)

# Sigmoid gate
topk_logits, topk_idx = router(x).topk(k)
weights = torch.sigmoid(topk_logits)
```

Renormalized Softmax can magnify small differences among selected experts. Sigmoid determines each weight independently and is used by DeepSeek-V3 and GLM-4.5.


In [ ]:
# === Dense FFN versus MoE: total and per-token active parameters ===
import torch
import matplotlib.pyplot as plt

d_model = 512
dense_d_ff = 2048
num_experts = 8
top_k = 2
expert_d_ff = dense_d_ff // top_k

# Ignore bias: one FFN contains an up-projection and a down-projection
dense_params = 2 * d_model * dense_d_ff
one_expert_params = 2 * d_model * expert_d_ff
moe_total_params = num_experts * one_expert_params
moe_active_params = top_k * one_expert_params

# Use four tokens to simulate the router's top-k selections
torch.manual_seed(42)
router_logits = torch.randn(4, num_experts)
selected_experts = router_logits.topk(top_k, dim=-1).indices

print("=== dense FFN → MoE ===")
print(f"Dense FFN parameters: {dense_params / 1e6:.2f}M, all used by every token")
print(f"MoE total parameters: {moe_total_params / 1e6:.2f}M")
print(f"MoE active parameters: {moe_active_params / 1e6:.2f}M per token")
print()
for token_id, experts in enumerate(selected_experts.tolist()):
    print(f"Token {token_id} selects experts: {experts}")
print()
print("Key observation: this configuration quadruples total FFN parameters,")
print("Yet each token activates roughly the same number of FFN parameters as the dense baseline.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(['Dense', 'MoE'], [dense_params, moe_total_params],
            color=['#64748b', '#2563eb'])
axes[0].set_title('Total FFN parameters')
axes[0].set_ylabel('Parameter count')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(['Dense', 'MoE'], [dense_params, moe_active_params],
            color=['#64748b', '#f59e0b'])
axes[1].set_title('Active parameters per token')
axes[1].set_ylabel('Parameter count')
axes[1].grid(True, axis='y', alpha=0.3)

for ax in axes:
    for bar in ax.patches:
        value = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, value, f'{value / 1e6:.2f}M',
                ha='center', va='bottom')
plt.tight_layout()
plt.show()


## 3. Position Representation and RoPE

Sinusoidal encoding adds a position vector to Token Embedding before Q and K projection, mixing content and positional terms in Attention scores. RoPE instead rotates Q and K: position $i$ rotates Q by $i	heta$ and position $j$ rotates K by $j	heta$, so their dot product naturally depends on relative offset $j-i$. We begin with 2D rotation, extend it to higher dimensions, and compare controlled Attention patterns.


In [ ]:
# === 2D rotation: mathematical intuition for RoPE ===
import matplotlib.pyplot as plt

import torch
import math

print("=== 2D rotation: why rotation can encode relative position ===")
print()

# A 2D vector
v = torch.tensor([1.0, 0.0])

# Rotation matrix
def rot_matrix(theta):
    """Rotate counter-clockwise by theta radians"""
    return torch.tensor([
        [math.cos(theta), -math.sin(theta)],
        [math.sin(theta),  math.cos(theta)],
    ])

# Rotate by 30 degrees and 60 degrees
theta_30 = math.radians(30)
theta_60 = math.radians(60)
R30, R60 = rot_matrix(theta_30), rot_matrix(theta_60)
v_30, v_60 = R30 @ v, R60 @ v

print(f"Original vector v:     ({v[0]:.1f}, {v[1]:.1f}), angle=0 deg")
print(f"After 30 deg rotation: ({v_30[0]:.3f}, {v_30[1]:.3f})")
print(f"After 60 deg rotation: ({v_60[0]:.3f}, {v_60[1]:.3f})")
print()

# Property 1: length preservation
print(f"Property 1 - length preservation: |v|={v.norm():.4f}, |R30*v|={v_30.norm():.4f}")
print("  Rotation doesn't change the vector's length, only its direction")

# Property 2: additivity
v_30_30 = R30 @ (R30 @ v)
print(f"Property 2 - additivity: R(30).R(30) = R(60)")
print(f"  R30·R30·v = ({v_30_30[0]:.3f}, {v_30_30[1]:.3f})")
print(f"  R60.v     = ({v_60[0]:.3f}, {v_60[1]:.3f})  <- consistent")

# Property 3 (most important): inner product depends only on the rotation angle difference
# (R(θ₁)u) · (R(θ₂)v) = u · R(θ₂-θ₁)v
u = torch.tensor([0.8, 0.6])
v2 = torch.tensor([0.3, 0.95])
dot_30_60 = (R30 @ u).dot(R60 @ v2)
dot_diff = u.dot(rot_matrix(theta_60 - theta_30) @ v2)
print(f"Property 3 - relativity (the core property RoPE relies on):")
print(f"  (R30·u)·(R60·v)     = {dot_30_60:.6f}")
print(f"  u.R(60-30).v        = {dot_diff:.6f}  <- depends only on angle difference 30 deg")
print()
print("-> Inner product of rotated vectors = original vectors' inner product rotated by the angle difference")
print("-> This means Q_i.K_j depends only on (j-i), not on the absolute values of i and j")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: rotation demo
for theta_deg, color, label in [(0, '#2c3e50', 'Original'),
                                 (30, '#e67e22', 'Rotated 30 deg'),
                                 (60, '#e74c3c', 'Rotated 60 deg')]:
    theta = math.radians(theta_deg)
    R = rot_matrix(theta)
    rotated = R @ v
    axes[0].quiver(0, 0, rotated[0].item(), rotated[1].item(), angles='xy',
                   scale_units='xy', scale=1, color=color, label=label,
                   width=0.018, headwidth=6)
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-0.5, 1.5)
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)
axes[0].set_title('2D rotation operation', fontsize=13, fontweight='bold')

# Right: how rotation encodes position
axes[1].set_xlim(0, 10); axes[1].set_ylim(0, 10)
axes[1].axis('off')
axes[1].text(5, 8.5, 'Core idea of RoPE', ha='center', fontsize=16, fontweight='bold')
axes[1].text(5, 7.0, 'Q at position i -> rotate by i theta', ha='center', fontsize=13)
axes[1].text(5, 6.2, 'K at position j -> rotate by j theta', ha='center', fontsize=13)
axes[1].text(5, 4.8, 'Q_i·K_j = f(j−i)', ha='center', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff3cd',
                      edgecolor='#f39c12', alpha=0.9))
axes[1].text(5, 3.0, 'Dot product depends on relative distance',
             ha='center', fontsize=12, color='#555')
axes[1].text(5, 2.0, 'Attention directly encodes token distance', ha='center', fontsize=13)

plt.tight_layout()
plt.show()


**RoPE in Higher Dimensions**

Q and K dimensions are paired, and each pair receives a separate 2D rotation frequency:

```text
pair 0: angle = pos / 10000^(0/d)
pair 1: angle = pos / 10000^(2/d)
pair 2: angle = pos / 10000^(4/d)
```

Fast low-dimensional frequencies distinguish nearby positions; slow high-dimensional frequencies represent longer ranges. Sinusoidal encoding uses these frequencies to create position vectors, whereas RoPE uses them to rotate Q and K.

| | Sinusoidal | RoPE |
|:---|:---|:---|
| Injection | add at input | rotate Q/K in Attention |
| Relative position | inferred from absolute encodings | encoded by angle difference |
| Extrapolation | computable but quality uncertain | computable but direct extrapolation can degrade |


In [ ]:
# === Full RoPE implementation ===

import torch

def precompute_rope_freqs(dim, seq_len, theta=10000.0):
    """
    Precompute cos and sin for each position and dimension pair

    Returns:
        cos: [seq_len, dim//2] - cos(pos * theta_k), theta_k = 1/(theta^(2k/dim))
        sin: [seq_len, dim//2] - sin(pos * theta_k)
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(seq_len).float()
    angles = torch.outer(positions, freqs)  # [seq_len, dim//2]
    return angles.cos(), angles.sin()

def apply_rotary_emb(x, cos, sin):
    """
    Apply rotary position encoding to x

    Pairs up the last dimension of x and applies 2D rotation to each pair:
        [x0, x1] -> [x0*cos - x1*sin, x1*cos + x0*sin]

    Args:
        x: [..., seq_len, d_head] - Q or K tensor
        cos, sin: [seq_len, d_head//2]
    Returns:
        Rotated tensor, same shape
    """
    d = x.shape[-1]
    x_reshaped = x.float().reshape(*x.shape[:-1], -1, 2)
    x1, x2 = x_reshaped[..., 0], x_reshaped[..., 1]

    # Align cos/sin to leading dimensions of x
    cos = cos.unsqueeze(0)
    sin = sin.unsqueeze(0)

    # 2D rotation
    rot1 = x1 * cos - x2 * sin
    rot2 = x2 * cos + x1 * sin

    rotated = torch.stack([rot1, rot2], dim=-1).flatten(-2)
    return rotated.type_as(x)

# === Hand calculation test: d_head=4 minimal example ===
print("=== RoPE hand calculation: d_head=4 ===")
print()

d_head = 4
seq_len = 3

# Same vector at three positions, to observe rotation effects
q = torch.tensor([[[1.0, 0.0, 0.5, 0.5],
                    [1.0, 0.0, 0.5, 0.5],
                    [1.0, 0.0, 0.5, 0.5]]])
print(f"Original Q (same vector at all three positions):")
for p in range(seq_len):
    print(f"  Position {p}: {[f'{v:.1f}' for v in q[0, p].tolist()]}")

# Compute RoPE cos/sin
cos, sin = precompute_rope_freqs(d_head, seq_len, theta=10000.0)
print(f"\ncos matrix [{cos.shape[0]} positions x {cos.shape[1]} dimension pairs]:")
print(cos)
print(f"\nsin matrix [{sin.shape[0]} positions x {sin.shape[1]} dimension pairs]:")
print(sin)

# Apply RoPE
q_rope = apply_rotary_emb(q, cos, sin)
print(f"\nQ after RoPE:")
for p in range(seq_len):
    vals = [f'{v:+.4f}' for v in q_rope[0, p].tolist()]
    print(f"  Position {p}: {vals}")

# Verify key property: same vector rotated, inner product decays with distance
print(f"\nVerifying 'inner product depends only on relative distance':")
dist_01 = (q_rope[0, 0] * q_rope[0, 1]).sum().item()
dist_02 = (q_rope[0, 0] * q_rope[0, 2]).sum().item()
dist_12 = (q_rope[0, 1] * q_rope[0, 2]).sum().item()
print(f"  Distance 1 (position 0 vs 1): {dist_01:.4f}")
print(f"  Distance 2 (position 0 vs 2): {dist_02:.4f}")
print(f"  Distance 1 (position 1 vs 2): {dist_12:.4f}")
print(f"  -> Same distance -> similar inner product; larger distance -> smaller inner product")
print(f"  -> This is the mechanism by which RoPE lets Attention naturally sense distance")


**Experiment: Three Position Representations**

All Tokens use nearly identical content vectors, so score differences come from position only. We compare no position encoding, sinusoidal encoding added before Q/K projection, and RoPE applied directly to projected Q/K. Observe how their Attention matrices differ.


In [ ]:
# === Experiment: comparing three position encodings' Attention patterns ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

print("=== Position encoding comparison experiment ===")
print("Design: all token vectors are nearly identical -> Attention differences come purely from position encoding")
print()

# Experiment parameters
seq_len = 16
d_model = 32
d_head = 32

# All tokens use nearly identical vectors
torch.manual_seed(42)
token_vec = torch.randn(1, d_model) * 0.5
x = token_vec.repeat(1, seq_len, 1)  # [1, 16, 32]
x = x + torch.randn(1, seq_len, d_model) * 0.03  # tiny noise to avoid singularity

# Reuse the sinusoidal encoding function from earlier
def get_sinusoidal_encoding(seq_len, d_model):
    """Sinusoidal position encoding (from earlier)"""
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

# Q/K projection (shared across all three approaches for fair comparison)
W_q = nn.Linear(d_model, d_head, bias=False)
W_k = nn.Linear(d_model, d_head, bias=False)
nn.init.xavier_normal_(W_q.weight, gain=0.5)
nn.init.xavier_normal_(W_k.weight, gain=0.5)

def compute_attention_maps(x_input):
    """
    Compute Attention score matrices for three approaches

    Returns: (noPE_scores, noPE_attn, sin_scores, sin_attn, RoPE_scores, RoPE_attn)
    """
    # === Approach 1: No position encoding ===
    q_raw = W_q(x_input)
    k_raw = W_k(x_input)
    scores_no = (q_raw @ k_raw.transpose(-2, -1)) / math.sqrt(d_head)
    attn_no = F.softmax(scores_no, dim=-1)

    # === Approach 2: Sinusoidal encoding (added to input) ===
    pe = get_sinusoidal_encoding(seq_len, d_model).unsqueeze(0)
    x_sin = x_input + pe
    q_sin = W_q(x_sin)
    k_sin = W_k(x_sin)
    scores_sin = (q_sin @ k_sin.transpose(-2, -1)) / math.sqrt(d_head)
    attn_sin = F.softmax(scores_sin, dim=-1)

    # === Approach 3: RoPE (rotating Q and K) ===
    q_rope = W_q(x_input)
    k_rope = W_k(x_input)
    cos, sin = precompute_rope_freqs(d_head, seq_len)
    q_rope = apply_rotary_emb(q_rope, cos, sin)
    k_rope = apply_rotary_emb(k_rope, cos, sin)
    scores_rope = (q_rope @ k_rope.transpose(-2, -1)) / math.sqrt(d_head)
    attn_rope = F.softmax(scores_rope, dim=-1)

    return (scores_no, attn_no, scores_sin, attn_sin, scores_rope, attn_rope)

scores_no, attn_no, scores_sin, attn_sin, scores_rope, attn_rope = compute_attention_maps(x)

# === Visualization: three columns x two rows ===
fig, axes = plt.subplots(2, 3, figsize=(19, 11))

titles_score = ['No Position Encoding\nAttention Score',
                'Sinusoidal Encoding\nAttention Score',
                'RoPE\nAttention Score']
titles_attn = ['No Position Encoding\nAttention Distribution (softmax)',
               'Sinusoidal Encoding\nAttention Distribution (softmax)',
               'RoPE\nAttention Distribution (softmax)']

for col, (t_s, t_a, s_mat, a_mat) in enumerate(zip(
    titles_score, titles_attn,
    [scores_no, scores_sin, scores_rope],
    [attn_no, attn_sin, attn_rope]
)):
    # First row: Score
    im0 = axes[0, col].imshow(s_mat[0].detach().numpy(), cmap='RdBu_r', aspect='auto')
    axes[0, col].set_title(t_s, fontsize=11, fontweight='bold')
    axes[0, col].set_xlabel('Key position'); axes[0, col].set_ylabel('Query position')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046)

    # Second row: Attention
    im1 = axes[1, col].imshow(a_mat[0].detach().numpy(), cmap='YlOrRd', aspect='auto')
    axes[1, col].set_title(t_a, fontsize=11, fontweight='bold')
    axes[1, col].set_xlabel('Key position'); axes[1, col].set_ylabel('Query position')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046)

plt.tight_layout()
plt.show()

# === Extract Attention's "distance preference" curves ===
print()
print("=== Attention distance preference analysis ===")

def distance_profile(attn_mat):
    """Compute average attention by |i-j|"""
    seq_len = attn_mat.shape[0]
    dists = {}
    for i in range(seq_len):
        for j in range(seq_len):
            d = abs(i - j)
            dists.setdefault(d, []).append(attn_mat[i, j].item())
    return [sum(dists[d]) / len(dists[d]) for d in sorted(dists)]

decay_no = distance_profile(attn_no[0])
decay_sin = distance_profile(attn_sin[0])
decay_rope = distance_profile(attn_rope[0])

fig, ax = plt.subplots(figsize=(10, 5))
dists = range(len(decay_no))
ax.plot(dists, decay_no, 'o-', label='No position encoding', linewidth=2, markersize=8)
ax.plot(dists, decay_sin, 's-', label='Sinusoidal encoding', linewidth=2, markersize=8)
ax.plot(dists, decay_rope, 'D-', label='RoPE', linewidth=2.5, markersize=9,
        color='#e74c3c')
ax.set_xlabel('Relative distance |i-j|', fontsize=13)
ax.set_ylabel('Average Attention weight', fontsize=13)
ax.set_title('Attention decay pattern with distance', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, len(dists) - 0.5)
plt.tight_layout()
plt.show()

# === Key observations ===
print()
print("Key observations (look at both heatmaps and distance curves together):")
print()
print("  1. No position encoding:")
print(f"     -> Attention is nearly uniform: distance 0 weight {decay_no[0]:.3f}")
print(f"       distance 15 weight {decay_no[-1]:.3f}")
print("     -> Because all token contents are nearly identical, the model has no position cues")
print()
print("  2. Sinusoidal position encoding:")
print(f"     -> Distance 0 weight {decay_sin[0]:.3f}")
print(f"       Distance 15 weight {decay_sin[-1]:.3f}")
print("     -> Position information is added to the input, so Q.K mixes in position-related terms")
print("     -> The model still needs to learn how to read relative distance from these terms")
print()
print("  3. RoPE:")
print(f"     -> Distance 0 weight {decay_rope[0]:.3f}")
print(f"       Distance 15 weight {decay_rope[-1]:.3f}")
print("     -> This small random experiment doesn't guarantee monotonic decay or a fixed pattern")
print("     -> What it aims to show is the mechanism: same relative distance corresponds to same rotation angle difference")
print()
print("RoPE's core advantage:")
print("  Not forcing Attention to decay with distance, but writing relative position information")
print("  directly into the inner product structure of Q and K, reducing the model's burden of learning distance mappings")


In [ ]:
# === Before and after: RoPE base 10,000 versus 1,000,000 ===
# A larger base rotates low-frequency dimension pairs more slowly, preserving distinctions in long sequences
import torch
import matplotlib.pyplot as plt
import math

d_head = 128
seq_len = 131072  # 128K

for base, color, label in [(10000, '#dc2626', 'old: base=10000'),
                            (1000000, '#2563eb', 'new: base=1000000')]:
    freqs = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    # Wavelength of each dimension pair = 2*pi/frequency
    wavelengths = (2 * math.pi) / freqs
    plt.semilogy(range(len(wavelengths)), wavelengths.tolist(),
                 'o-', color=color, linewidth=1.5, markersize=3, label=label)

plt.xlabel('Dimension pair index (0=high freq, 63=low freq)')
plt.ylabel('Wavelength (tokens per full rotation)')
plt.title('RoPE base: old 10k vs new 1M wavelength distribution')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.axhline(seq_len, color='gray', linestyle='--', linewidth=1)
plt.text(2, seq_len * 1.3, f'128K context', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

print("Key observations:")
print("  Old (base=10000): low-frequency pairs have already made dozens of turns by position 128K,")
print("                    wrapping position information and making distant locations ambiguous")
print("  New (base=1000000): low-frequency pairs have not completed one turn by position 128K,")
print("                      so positions remain distinguishable when extrapolating to long sequences")
print()
print("  Models that changed it: Qwen3 (1M), GLM-4.5 (1M), Gemma3 global layers (1M), MiniMax (10M)")
print("  Model that did not: LLaMA 2 (10k), which relies on YaRN for extrapolation")


**NoPE and iRoPE: Layers Without Position Encoding**

| | RoPE in Every Layer | Some NoPE Layers |
|:---|:---|:---|
| Method | rotate every Q/K | alternate RoPE and no position encoding |
| Assumption | position must be explicit | deep Attention can infer relative position |
| Extrapolation | depends on base scaling / YaRN | NoPE layers are length-independent |
| Examples | traditional RoPE models | Llama 4 iRoPE, Jamba, SmolLM3 |

Llama 4 uses RoPE layers for precise local distances and NoPE layers for global processing. This less common design is still being evaluated across model families.


## 4. KV Cache and Shared Attention Heads

When generating Token 4097, the model reads cached K and V for the previous 4096 positions rather than recomputing them. At long contexts this KV Cache can exceed model-weight memory. MHA, GQA, and MLA retain Attention's read operation but organize and store historical K/V differently. We compare their head counts and cache shapes.


**MHA and GQA**

The previous four improvements addressed stability, effectiveness, efficiency, and position encoding. Attention still has one engineering issue: the multi-head structure is memory-hungry at inference.

Every head must cache its own K and V (the KV Cache); the more heads, the larger the memory. For a 32-head model, the KV Cache holds 32 copies of K and 32 copies of V. When the context length stretches to thousands or even tens of thousands, this memory easily becomes the bottleneck.

GQA (Grouped-Query Attention) lets multiple Q heads share the same group of K and V, thereby shrinking the KV Cache.


In [ ]:
# === MHA vs GQA vs MQA: KV Cache hand calculation comparison ===
print("=== KV Cache comparison: MHA / GQA / MQA ===")
print()

# Assume: a decoder-only model with 32 Q heads, 128 dims per head
num_q_heads = 32      # number of Q heads
d_head = 128          # dimension per head
seq_len = 4096        # sequence length
dtype_bytes = 2       # bf16 takes 2 bytes

# KV head counts for the three approaches
configs = [
    ("MHA", num_q_heads, num_q_heads),
    ("GQA-4", 4, num_q_heads),
    ("GQA-8", 8, num_q_heads),
    ("MQA", 1, num_q_heads),
]

print(f"Model config: {num_q_heads} Q heads, {d_head} dims per head")
print(f"Sequence length: {seq_len}, dtype: bf16 (2 bytes)")
print()

mha_size = None
cache_names = []
cache_mb = []

for name, kv_heads, _ in configs:
    # KV Cache size = 2(K+V) * kv_heads * d_head * seq_len * dtype_bytes
    kv_size = 2 * kv_heads * d_head * seq_len * dtype_bytes
    kv_size_mb = kv_size / (1024 ** 2)
    
    if mha_size is None:
        mha_size = kv_size
    ratio = kv_size / mha_size * 100
    cache_names.append(name)
    cache_mb.append(kv_size_mb)
    
    # How many Q heads per group
    q_per_group = num_q_heads // kv_heads
    
    print(f"{name:<8s} KV heads={kv_heads:>2d}  "
          f"Q/group={q_per_group:>2d}  "
          f"KV Cache={kv_size_mb:>6.1f} MB  "
          f"({ratio:>5.1f}%)")

print()
print("Key numbers:")
print(f"  MHA:   {2 * num_q_heads * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB (32 KV heads)")
print(f"  GQA-4: {2 * 4 * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB (4 KV heads, saves 87.5%)")
print(f"  MQA:   {2 * 1 * d_head * seq_len * dtype_bytes / (1024**2):.0f} MB (1 KV head, saves ~97%)")
print()
print("Real examples: LLaMA 2 70B uses GQA; Mistral 7B and LLaMA 3 also use GQA")
print("Note: LLaMA 2's 7B/13B versions are not this configuration, don't mix up model sizes")
print("Intuition: KV cache shrinks roughly proportional to how much you reduce the KV head count")

plt.figure(figsize=(7, 3.8))
bars = plt.bar(cache_names, cache_mb, color=['#64748b', '#2563eb', '#60a5fa', '#f59e0b'])
plt.ylabel('KV cache per layer (MB)')
plt.title('Fewer KV heads, smaller KV cache')
plt.grid(True, axis='y', alpha=0.3)
for bar, value in zip(bars, cache_mb):
    plt.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.1f}',
             ha='center', va='bottom')
plt.tight_layout()
plt.show()

**GQA implementation**

The only difference between GQA and MHA is one step: split Q into "groups," and the Q heads within each group share the same K and V.

```
MHA:  Q [32, d] x K [32, d] -> 32 independent attention groups
GQA:  Q [32, d] x K [4, d]  -> 32 Q heads split into 4 groups, each group of 8 Q heads shares 1 K

Implementation:
  1. K/V only have 4 heads, expanded to 32 heads via repeat
  2. Then do attention exactly the same as MHA
```

So the code change for GQA is minimal - just one extra `expand` on K and V.


In [ ]:
# === GQA implementation demo ===

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA)

    Args:
        d_model: model dimension
        num_q_heads: number of Q heads (e.g., 32)
        num_kv_heads: number of K/V heads (e.g., 4), must divide num_q_heads
    """
    def __init__(self, d_model, num_q_heads, num_kv_heads):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_head = d_model // num_q_heads
        self.q_per_group = num_q_heads // num_kv_heads  # how many Q heads per group

        # Q has num_q_heads heads, K/V only have num_kv_heads heads
        self.W_q = nn.Linear(d_model, num_q_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(num_q_heads * self.d_head, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape

        # Projection
        q = self.W_q(x)  # [B, S, num_q * d_head]
        k = self.W_k(x)  # [B, S, num_kv * d_head]
        v = self.W_v(x)  # [B, S, num_kv * d_head]

        # Reshape into multi-head format
        q = q.view(B, S, self.num_q_heads, self.d_head)
        k = k.view(B, S, self.num_kv_heads, self.d_head)
        v = v.view(B, S, self.num_kv_heads, self.d_head)

        # GQA key step: expand K/V to the same number of heads as Q
        # Each KV head is shared by self.q_per_group Q heads
        k = k[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)

        v = v[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)

        # Transpose to [B, heads, S, d_head] for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot-product attention (same as before)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        # Merge heads, output projection
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)

# Test: compare MHA and GQA
d_model = 256
num_q = 8    # 8 Q heads
num_kv = 2   # 2 KV heads -> each group of 4 Q heads shares 1 KV

gqa = GroupedQueryAttention(d_model, num_q, num_kv)
mha = nn.MultiheadAttention(d_model, num_q, batch_first=True)

x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)

out_gqa = gqa(x, mask)
out_mha, _ = mha(x, x, x, attn_mask=mask, need_weights=False)

print("=== GQA implementation ===")
print(f"Input: {x.shape}")
print(f"GQA output: {out_gqa.shape}")
print(f"MHA output: {out_mha.shape}")
print()

# Parameter count comparison
gqa_params = sum(p.numel() for p in gqa.parameters())
mha_params = sum(p.numel() for p in mha.parameters())
print(f"GQA params: {gqa_params:,}  (smaller KV projections)")
print(f"MHA params: {mha_params:,}")
print(f"GQA params are {gqa_params/mha_params:.2%} of MHA")
print()
print(f"KV projection comparison:")
print(f"  MHA: W_k {d_model}x{d_model} = {d_model*d_model:,}")
print(f"  GQA: W_k {d_model}x{num_kv * (d_model//num_q)} = {d_model * num_kv * (d_model//num_q):,}")
print(f"  -> GQA's KV projection is only {num_kv/num_q:.0%} of MHA's")


**Low-Rank KV Representation in MLA**


In [ ]:
# === MHA vs GQA vs MLA: KV Cache comparison ===
print("=== KV Cache three-stage comparison: MHA -> GQA -> MLA ===")
print()

# Small model parameters
d_model = 512
num_q_heads = 8
d_head = d_model // num_q_heads  # = 64
d_latent = 64  # MLA compression dimension (much smaller than d_model=512)
seq_len = 4096
dtype_bytes = 2  # bf16

print(f"Model: d_model={d_model}, Q heads={num_q_heads}, d_head={d_head}")
print(f"Sequence length: {seq_len}, dtype: bf16")
print(f"MLA latent dimension: {d_latent}")
print()

# MHA: KV Cache = 2 × num_heads × d_head × seq_len × bytes
mha_kv = 2 * num_q_heads * d_head * seq_len * dtype_bytes

# GQA-4: 4 KV heads
gqa_kv_heads = 4
gqa_kv = 2 * gqa_kv_heads * d_head * seq_len * dtype_bytes

# GQA-2: 2 KV heads
gqa2_kv = 2 * 2 * d_head * seq_len * dtype_bytes

# MLA: only store compressed latent
mla_kv = d_latent * seq_len * dtype_bytes  # simplified assumption: only store joint latent c_KV

configs = [
    ("MHA", mha_kv),
    ("GQA-4", gqa_kv),
    ("GQA-2", gqa2_kv),
    ("MLA", mla_kv),
]

print(f"{'Method':<10s} {'KV Cache size':>16s} {'vs MHA':>10s} {'Note'}")
print("-" * 65)

for name, size in configs:
    mb = size / (1024**2)
    ratio = size / mha_kv * 100
    note = ""
    if name == "MHA":
        note = "Each Q head has independent KV"
    elif name == "GQA-4":
        note = "4 groups of Q share KV"
    elif name == "GQA-2":
        note = "Each Q head has independent KV"
    elif name == "MLA":
        note = f"Low-rank compression to {d_latent} dims"
    print(f"{name:<10s} {mb:>13.1f} MB {'(' + str(int(ratio)) + '%)':>11s} {note}")

print()
print("Key numbers:")
print(f"  MHA -> GQA-4: saves {(1 - gqa_kv/mha_kv)*100:.0f}%")
print(f"  MHA -> MLA:   saves {(1 - mla_kv/mha_kv)*100:.0f}%")
print(f"  GQA-4 -> MLA: saves {(1 - mla_kv/gqa_kv)*100:.0f}%")
print()
print("MLA intuition: d_latent << d_model (64 << 512), Cache can be significantly smaller")
print("Real MLA also handles RoPE decoupling and other details; here we focus on the core low-rank compression idea")
print("The premise for low-rank compression: K/V information has significant redundancy that can be approximated with fewer dimensions")

seq_lengths = torch.tensor([4096, 32768, 131072], dtype=torch.float64)
mha_curve = 2 * num_q_heads * d_head * seq_lengths * dtype_bytes / (1024 ** 2)
gqa_curve = 2 * gqa_kv_heads * d_head * seq_lengths * dtype_bytes / (1024 ** 2)
mla_curve = d_latent * seq_lengths * dtype_bytes / (1024 ** 2)

plt.figure(figsize=(7, 4))
plt.plot(seq_lengths, mha_curve, marker='o', label='MHA')
plt.plot(seq_lengths, gqa_curve, marker='o', label='GQA-4')
plt.plot(seq_lengths, mla_curve, marker='o', label='MLA (teaching simplification)')
plt.xlabel('Context length')
plt.ylabel('KV cache per layer (MB)')
plt.title('KV cache growth with context length')
plt.xticks(seq_lengths, ['4K', '32K', '128K'])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

**MLA implementation**

The difference between MLA and standard Attention is only in where KV comes from:

```text
Standard: K = W_K @ x,  V = W_V @ x  (project directly from x)
MLA:  c_KV = W_down @ x          (first compress to a small latent)
      K = W_up_K @ c_KV          (at inference, decompress latent into K)
      V = W_up_V @ c_KV          (at inference, decompress latent into V)
```

Below is a minimal implementation showing this mechanism. The full MLA also involves RoPE decoupling; here we focus on the core low-rank compression idea.


In [ ]:
# === Minimal MLA implementation (showing core low-rank compression mechanism) ===

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadLatentAttention(nn.Module):
    """
    MLA (Multi-head Latent Attention) minimal version

    KV is first compressed into a d_latent-dimensional latent space,
    then decompressed into K and V at inference time.

    Args:
        d_model: model dimension
        num_heads: number of attention heads
        d_latent: KV compression dimension (much smaller than d_model)
    """
    def __init__(self, d_model, num_heads, d_latent):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent

        # Q projection as usual
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        # KV first compresses to latent
        self.W_kv_down = nn.Linear(d_model, d_latent, bias=False)
        # Then decompresses from latent into K and V
        self.W_k_up = nn.Linear(d_latent, d_model, bias=False)
        self.W_v_up = nn.Linear(d_latent, d_model, bias=False)
        # Output projection
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape

        # Q: normal projection
        q = self.W_q(x).view(B, S, self.num_heads, self.d_head)

        # KV: first compress to latent, then decompress
        c_kv = self.W_kv_down(x)           # [B, S, d_latent]
        k = self.W_k_up(c_kv).view(B, S, self.num_heads, self.d_head)
        v = self.W_v_up(c_kv).view(B, S, self.num_heads, self.d_head)

        # Standard scaled dot-product attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(out), c_kv  # return c_kv as the KV Cache

# Test MLA
d_model = 64
num_heads = 4
d_latent = 16  # compression dimension: 16 << 64

mla = MultiHeadLatentAttention(d_model, num_heads, d_latent)
x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)

out, c_kv = mla(x, mask)

print("=== MLA minimal implementation test ===")
print(f"d_model={d_model}, num_heads={num_heads}, d_latent={d_latent}")
print(f"Input: {x.shape}")
print(f"Compressed latent c_kv: {c_kv.shape}  <- this is the KV Cache size")
print(f"Output: {out.shape}")
print()

# KV Cache comparison
mha_kv_per_token = 2 * num_heads * (d_model // num_heads)  # K+V full dimension
mla_kv_per_token = d_latent  # only store latent

print(f"KV Cache per token (element count):")
print(f"  MHA: {mha_kv_per_token} (K+V, {num_heads} heads x {d_model//num_heads} dims)")
print(f"  MLA: {mla_kv_per_token} (only c_kv, {d_latent} dims)")
print(f"  Compression ratio: {mla_kv_per_token/mha_kv_per_token:.1%}")
print()
print("MLA trade-off:")
print(f"  Training: extra W_kv_down, W_k_up, W_v_up linear layers (additional params and compute)")
print(f"  Inference: KV Cache drops from {mha_kv_per_token} to {mla_kv_per_token} elements")
print(f"  -> Same memory can handle {(mha_kv_per_token/mla_kv_per_token):.0f}x longer sequences")


## 5. Long-Context Attention

KV Cache avoids repeated computation but global Attention still contains $N^2$ score positions. Local and sparse methods read fewer positions; linear Attention, DeltaNet, KDA, and Mamba compress history into fixed-size state.

Standard Attention computes $(QK^T)V$, materializing an $N	imes N$ matrix. Reassociation gives $Q(K^TV)$ with a $d	imes d$ intermediate and complexity $O(Nd^2)$ rather than $O(N^2)$. However, Softmax is nonlinear, so this is not algebraically identical. Linear methods replace Softmax with feature maps or state-update rules and may lose exact long-range retrieval ability.


In [ ]:
# === Hand-calculate linear attention before and after reassociation ===
import torch

torch.manual_seed(42)
N, d = 4, 3  # Tiny example: four tokens and three dimensions per head

Q = torch.randn(N, d)
K = torch.randn(N, d)
V = torch.randn(N, d)

# Old: standard Attention computes (Q @ K^T) @ V
attn_matrix = Q @ K.T          # [N, N] = [4, 4]
# Omit softmax and focus on matrix shapes
old_out = attn_matrix @ V      # [N, d] = [4, 3]

# New: linear Attention computes Q @ (K^T @ V)
kv_matrix = K.T @ V            # [d, d] = [3, 3]
new_out = Q @ kv_matrix        # [N, d] = [4, 3]

print("=== Parameter computation ===")
print(f"Q shape: {Q.shape},  K shape: {K.shape},  V shape: {V.shape}")
print()
print(f"Old (standard): compute QxK^T as a [{N}x{N}] matrix, then multiply by V")
print(f"  Intermediate attn = QxK^T, shape = {attn_matrix.shape}")
print(f"  Number of elements in the NxN matrix = {N*N}")
print(f"  Output = attnxV, shape = {old_out.shape}")
print()
print(f"New (linear): compute K^TxV as a [{d}x{d}] matrix, then multiply by Q")
print(f"  Intermediate kv = K^TxV, shape = {kv_matrix.shape}")
print(f"  Number of elements in the dxd matrix = {d*d}")
print(f"  Output = Qxkv, shape = {new_out.shape}")
print()
print(f"How much smaller? N^2={N*N} versus d^2={d*d}")
print(f"  With N=131072 (128K) and d=128: N^2={131072**2:,} versus d^2={128**2:,}")
print(f"  The intermediate matrix is {131072**2 / 128**2:,.0f}x smaller")
print()
print("Note: softmax is omitted here. Real linear attention needs a feature map or a state-update rule.")
print("This experiment therefore compares intermediate sizes; it does not claim numerical equivalence.")

context_lengths = torch.tensor([1024, 4096, 16384, 65536, 131072],
                               dtype=torch.float64)
head_dim = 128
softmax_elements = context_lengths ** 2
linear_state_elements = torch.full_like(context_lengths, head_dim ** 2)

plt.figure(figsize=(7, 4))
plt.loglog(context_lengths, softmax_elements, marker='o', label='N x N attention matrix')
plt.loglog(context_lengths, linear_state_elements, marker='o',
           label='d x d teaching state')
plt.xlabel('Context length N')
plt.ylabel('Intermediate elements (log scale)')
plt.title('Intermediate state size as context grows')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


**Hybrid Linear and Softmax Attention**

Pure linear Attention is efficient but weaker at exact retrieval; pure Softmax is quadratic. Hybrid models use mostly state-based layers and periodically insert global Attention.

| Model | Linear:Softmax | Global Share |
|:---|:---|:---|
| GPT-2 / LLaMA | 0:1 | 100% |
| MiniMax-01 | 7:1 | 12.5% |
| Jamba | 7:1 | 12.5% |
| Qwen3.8 | 3:1 | 25% |
| Kimi K3 | 69:24 | about 25.8% |

The ratio trades computation for retrieval: state layers maintain continuous memory, while global layers read distant information precisely.


**Alternating Local and Global Attention**

| | All Global | Local / Global Alternation |
|:---|:---|:---|
| Per-layer view | all $N$ Tokens | local window in most layers |
| KV Cache | all history in every layer | window only in local layers |
| Global information | every layer | periodic global layers |
| Example | traditional models | Gemma 3 uses 5 local + 1 global |

Gemma 3's local layers cache only 1024 Tokens, so savings grow with context length. The trade-off is that distant information must travel through several local layers.


In [ ]:
# === Full, local, and sparse attention: which positions can each token read? ===
import torch
import matplotlib.pyplot as plt

seq_len = 24
window = 5
row = torch.arange(seq_len)[:, None]
col = torch.arange(seq_len)[None, :]
causal = col <= row

full_mask = causal
local_mask = causal & (col >= row - window + 1)
# Teaching pattern: outside the local window, retain one global anchor every six positions
sparse_mask = local_mask | (causal & (col % 6 == 0))

last_token = seq_len - 1
print("=== One token through FFN ===")
print(f"Full Attention:  {int(full_mask[last_token].sum())}")
print(f"Local Attention: {int(local_mask[last_token].sum())}")
print(f"Sparse example:  {int(sparse_mask[last_token].sum())}")
print("Key observation: local and sparse attention both read fewer positions, but retain distant information differently.")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, mask, title in zip(
    axes,
    [full_mask, local_mask, sparse_mask],
    ['Full causal', 'Sliding window', 'Sparse teaching pattern'],
):
    ax.imshow(mask.float(), cmap='Blues', origin='upper', aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Key position')
axes[0].set_ylabel('Query position')
plt.suptitle('Positions available to each query token')
plt.tight_layout()
plt.show()


## 6. Training Objectives and Optimization

Architecture changes still require stable learning. MTP changes the objective, QK-Norm changes Attention, and QK-Clip and Muon act during parameter updates.

| | NTP | NTP + MTP |
|:---|:---|:---|
| Prediction | Token $i+1$ | Tokens $i+1$ and $i+2$ |
| Signal | one Cross-Entropy | two weighted losses |
| Extra structure | none | auxiliary Block with shared embedding/head |
| Inference | standard | discard MTP or use it to draft Tokens |
| Examples | GPT-2, LLaMA | DeepSeek-V3, GLM-4.5 |

MTP adds a farther-future target to the same hidden state. The auxiliary module can be removed at inference or used for speculative proposals, whose speedup depends on acceptance rate.


In [ ]:
# === Hand-calculate MTP loss: predict token i+1 and token i+2 together ===
import torch
import torch.nn.functional as F

# Tiny example: sequence [5, 12, 3] with vocabulary size 30
input_ids = torch.tensor([5, 12, 3])
vocab_size = 30

# Simulated logits: the main head predicts i+1 and the MTP head predicts i+2
torch.manual_seed(42)
main_logits = torch.randn(3, vocab_size)   # Positions 0, 1, and 2 each predict the next token
mtp_logits = torch.randn(3, vocab_size)    # Positions 0, 1, and 2 each predict the token after next

# NTP loss: position i predicts input_ids[i+1]
# Position 0 predicts 12; position 1 predicts 3; position 2 has no next token and is ignored
ntp_targets = input_ids[1:]  # [12, 3]
ntp_loss = F.cross_entropy(main_logits[:2], ntp_targets)

# MTP loss: position i predicts input_ids[i+2]
# Position 0 predicts 3; position 1 has no token two steps ahead and is ignored
mtp_targets = input_ids[2:]  # [3]
mtp_loss = F.cross_entropy(mtp_logits[:1], mtp_targets)

# Total loss = NTP + lambda * MTP
lam = 0.3
total_loss = ntp_loss + lam * mtp_loss

print("=== Hand-calculate MTP loss ===")
print(f"Sequence: {input_ids.tolist()}")
print(f"Vocabulary size: {vocab_size}")
print()
print(f"NTP: position 0 predicts {ntp_targets[0].item()}, position 1 predicts {ntp_targets[1].item()}")
print(f"  NTP loss = {ntp_loss.item():.4f}")
print()
print(f"MTP: position 0 predicts {mtp_targets[0].item()} by skipping one token")
print(f"  MTP loss = {mtp_loss.item():.4f}")
print()
print(f"Total loss = NTP + lambda*MTP = {ntp_loss.item():.4f} + {lam}*{mtp_loss.item():.4f}")
print(f"        = {total_loss.item():.4f}")
print()
print("Key observations:")
print(f"  Old (NTP only): each position learns one-token-ahead prediction, loss = {ntp_loss.item():.4f}")
print(f"  New (NTP+MTP): adds a farther-ahead prediction signal, loss = {total_loss.item():.4f}")
print("  lambda=0.3 gives the auxiliary objective a loss weight of 0.3.")

demo_seq_len = 8
ntp_target_count = demo_seq_len - 1
mtp_extra_count = demo_seq_len - 2

plt.figure(figsize=(6, 3.8))
plt.bar(['NTP', 'NTP + MTP'], [ntp_target_count, ntp_target_count],
        color='#64748b', label='Next-token targets')
plt.bar(['NTP', 'NTP + MTP'], [0, mtp_extra_count],
        bottom=[ntp_target_count, ntp_target_count],
        color='#2563eb', label='Extra future targets')
plt.ylabel('Supervised target pairs')
plt.title('Training signals from an 8-token sequence')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


**QK-Norm**


In [ ]:
# === QK-Norm comparison: without vs with ===
import torch
import torch.nn.functional as F
import math

print("=== QK-Norm: simulating Attention logit degradation ===")
print()

# Simulate Q and K from late training: large magnitude differences
torch.manual_seed(42)
d_head = 8
seq_len = 4

# Normal Q
Q_normal = torch.randn(1, 1, seq_len, d_head) * 1.0
# K with large magnitude from late training
K_large = torch.randn(1, 1, seq_len, d_head) * 8.0

# Without QK-Norm: compute attention score directly
score_no_norm = (Q_normal @ K_large.transpose(-2, -1)) / math.sqrt(d_head)
attn_no_norm = F.softmax(score_no_norm, dim=-1)

# With QK-Norm: apply RMSNorm to Q and K separately first
Q_normed = Q_normal / torch.sqrt(torch.mean(Q_normal ** 2, dim=-1, keepdim=True) + 1e-6)
K_normed = K_large / torch.sqrt(torch.mean(K_large ** 2, dim=-1, keepdim=True) + 1e-6)
score_with_norm = (Q_normed @ K_normed.transpose(-2, -1)) / math.sqrt(d_head)
attn_with_norm = F.softmax(score_with_norm, dim=-1)

print("Without QK-Norm:")
print(f"  Q magnitude range: {Q_normal.norm(dim=-1).min():.2f} ~ {Q_normal.norm(dim=-1).max():.2f}")
print(f"  K magnitude range: {K_large.norm(dim=-1).min():.2f} ~ {K_large.norm(dim=-1).max():.2f}")
print(f"  Attention score: {score_no_norm[0,0,0].tolist()}")
print(f"  Attention distribution:  {[f'{v:.3f}' for v in attn_no_norm[0,0,0].tolist()]}")
print(f"  -> softmax degradation: most probability concentrates on individual positions")

print()
print("With QK-Norm:")
print(f"  Q magnitude range: {Q_normed.norm(dim=-1).min():.2f} ~ {Q_normed.norm(dim=-1).max():.2f}")
print(f"  K magnitude range: {K_normed.norm(dim=-1).min():.2f} ~ {K_normed.norm(dim=-1).max():.2f}")
print(f"  Attention score: {score_with_norm[0,0,0].tolist()}")
print(f"  Attention distribution:  {[f'{v:.3f}' for v in attn_with_norm[0,0,0].tolist()]}")
print(f"  -> Distribution is no longer extremely concentrated; Attention can aggregate from multiple positions")

print()
print("Key observation: QK-Norm brings the RMS of each Q and K vector close to one,")
print("significantly narrowing the score scale, so softmax doesn't easily degrade.")

positions = torch.arange(seq_len)
width = 0.36
plt.figure(figsize=(7, 3.8))
plt.bar(positions - width / 2, attn_no_norm[0, 0, 0].detach(), width,
        label='Without QK-Norm', color='#dc2626')
plt.bar(positions + width / 2, attn_with_norm[0, 0, 0].detach(), width,
        label='With QK-Norm', color='#2563eb')
plt.xlabel('Key position')
plt.ylabel('Attention probability')
plt.title('QK-Norm changes softmax concentration')
plt.xticks(positions)
plt.grid(True, axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# === Full GQA implementation with QK-Norm ===

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttention_QKNorm(nn.Module):
    """
    GQA + QK-Norm (a technique for stabilizing attention logits)

    Compared to standard GQA, only two extra RMSNorm lines: one for Q, one for K.
    """
    def __init__(self, d_model, num_q_heads, num_kv_heads, eps=1e-6):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.d_head = d_model // num_q_heads
        self.q_per_group = num_q_heads // num_kv_heads

        self.W_q = nn.Linear(d_model, num_q_heads * self.d_head, bias=False)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head, bias=False)
        self.W_o = nn.Linear(num_q_heads * self.d_head, d_model, bias=False)

        # QK-Norm: apply RMSNorm to Q and K separately
        self.q_norm = RMSNorm(self.d_head, eps=eps)
        self.k_norm = RMSNorm(self.d_head, eps=eps)

    def forward(self, x, mask=None):
        B, S, D = x.shape

        q = self.W_q(x).view(B, S, self.num_q_heads, self.d_head)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_head)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_head)

        # QK-Norm: apply RMSNorm to Q and K before computing attention
        q = self.q_norm(q)
        k = self.k_norm(k)

        # GQA expand
        k = k[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)
        v = v[:, :, None, :, :].expand(
            B, S, self.q_per_group, self.num_kv_heads, self.d_head
        ).reshape(B, S, self.num_q_heads, self.d_head)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores + mask
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.W_o(out)

# Test
d_model, num_q, num_kv = 16, 4, 2
attn_qknorm = GroupedQueryAttention_QKNorm(d_model, num_q, num_kv)
x = torch.randn(1, 6, d_model)
mask = torch.triu(torch.ones(6, 6) * float('-inf'), diagonal=1)
out = attn_qknorm(x, mask)

print("=== GQA + QK-Norm test ===")
print(f"Input: {x.shape}, Output: {out.shape}")
print(f"Q heads: {num_q}, KV heads: {num_kv}")
print(f"Change: two extra lines - self.q_norm and self.k_norm compared to standard GQA")
print("Effect: Q and K scales are more controlled, attention logits are less likely to grow abnormally")


**QK-Clip: Post-Update Control of Attention Logits**

QK-Norm normalizes Q and K during every forward pass. QK-Clip instead checks each head after an optimizer step. If its maximum logit $S_{max}^h$ exceeds threshold $	au$, scale both Q and K by $\sqrt{\gamma_h}$, where $\gamma_h=\min(1,	au/S_{max}^h)$. This brings excessive logits under the threshold without changing their relative ordering. QK-Norm acts continuously; QK-Clip intervenes only after a violation.


**AdamW and Muon Optimizers**

AdamW uses momentum and second-moment adaptive scaling. Muon applies Newton–Schulz orthogonalization to gradient matrix $G$, approximating $G(G^TG)^{-1/2}$ so update energy is distributed more evenly across singular directions. Kimi K2's MuonClip combines Muon with QK-Clip, while GLM-4.5 also uses Muon for most matrix parameters. The essential point here is that Muon changes the update geometry; the full Newton–Schulz derivation is optional.


## 7. Designing a Modern Language-Model Architecture

Use six questions to read any architecture: where Norm and residuals sit, how position is represented, how KV is stored, how long context is read, how FFN capacity grows, and how training signal is supplied. GPT-2 combines Pre-Norm, LayerNorm, GELU, MHA, and learned positions. LLaMA uses RMSNorm, SwiGLU, RoPE, and later GQA. DeepSeek-V3 uses MLA, MoE, and MTP. Newer systems choose sparse retrieval or hybrid recurrent-state/global-Attention designs. Names change; the component questions remain useful.


**Combining Components in a LLaMA-Style Block**

```text
LLaMA Block = Pre-Norm RMSNorm + RoPE Attention + Pre-Norm RMSNorm + SwiGLU FFN

x → RMSNorm → RoPE Attention → residual
  → RMSNorm → SwiGLU FFN → residual → output
```


In [ ]:
import torch
import torch.nn as nn

class LLaMABlock(nn.Module):
    """
    Modern LLM Transformer Block

    Three major upgrades vs earlier:
    1. LayerNorm -> RMSNorm
    2. Post-Norm -> Pre-Norm
    3. ReLU FFN -> SwiGLU FFN
    """
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        # Pre-Norm: RMSNorm before Attention
        self.norm_attn = RMSNorm(d_model)
        # Multi-Head Self-Attention (same as before)
        self.attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        # Pre-Norm: RMSNorm before FFN
        self.norm_ffn = RMSNorm(d_model)
        # SwiGLU FFN
        self.ffn = FeedForward_SwiGLU(d_model, d_ff)

    def forward(self, x, mask=None):
        # Note the order: Norm first, then sublayer, then residual
        h = self.norm_attn(x)
        attn_out, _ = self.attention(h, h, h, attn_mask=mask, need_weights=False)
        x = x + attn_out  # residual: bypasses Norm!

        h = self.norm_ffn(x)
        x = x + self.ffn(h)  # residual: bypasses Norm!

        return x

# Test
d_model, num_heads = 8, 2
block_new = LLaMABlock(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)
causal_mask = torch.triu(torch.ones(5, 5) * float('-inf'), diagonal=1)

out_new = block_new(test_x, causal_mask)
print("=== LLaMA Block test ===")
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {out_new.shape}  <- unchanged!")
print(f"But all internal components have been upgraded:")
print(f"  \✓ RMSNorm replaces LayerNorm")
print(f"  \✓ Pre-Norm replaces Post-Norm")
print(f"  \✓ SwiGLU replaces ReLU FFN")


## Summary

Confirm you understand these (check in order):

1. \✅ LayerNorm = (x - mu) / sigma * gamma + beta, computes two statistics
2. \✅ RMSNorm = x / RMS(x) * gamma, computes only one statistic - removes mean subtraction
3. \✅ What RMSNorm saves: no need to compute mu, no need to compute (x-mu)^2 (uses x^2 directly)
4. \✅ Sinusoidal encoding is added to the input, RoPE rotates Q and K - different ways of injecting position into Attention
5. \✅ RoPE's core property: Q_i.K_j depends only on relative position (j-i)
6. \✅ RoPE implementation: pair up dimensions for 2D rotation, low-dim high-frequency, high-dim low-frequency
7. \✅ The FFN is the per-token processing workshop inside the Block; modern models heavily upgrade it
8. \✅ ReLU's problem: negatives become 0, gradient in the negative region is also 0
9. \✅ SwiGLU = information channel (W_up) * gating channel (Swish(W_gate))
10. \✅ Swish is smooth with non-zero gradients in the negative region, unlike ReLU's hard cutoff
11. \✅ Post-Norm = sublayer+residual->Norm, residual path goes through Norm
12. \✅ Pre-Norm = Norm->sublayer->+residual, residual path bypasses Norm
13. \✅ Why Pre-Norm is better: residual path is more direct, deep training is typically more stable
14. \✅ LLaMA-style Block = Pre-Norm + RMSNorm + RoPE + SwiGLU
15. \✅ GQA = multiple Q heads share one set of KV heads, KV Cache is significantly reduced
16. \✅ MHA (32 KV heads) -> GQA (4 KV heads, saves 87.5%) -> MQA (1 KV head, saves ~97%)
17. \✅ GQA implementation key: smaller K/V projections, expand to match Q heads before attention
18. \✅ QK-Norm = normalize Q and K separately before computing attention, controlling logit scale
19. \✅ MLA = low-rank KV compression to latent space, compress to smaller c_KV first, decompress at inference
20. \✅ MLA can significantly reduce KV Cache, but real implementation also involves RoPE decoupling and other details

**One-sentence summary**: Modern Decoder-only models don't overthrow GPT-2; they upgrade key components on the same backbone - Pre-Norm for deep-training stability, RoPE for better position encoding, SwiGLU for enhanced FFN, RMSNorm for simplified normalization, GQA for reduced KV Cache, QK-Norm for controlling attention logit scale, and MLA for further compressing KV information.


## Exercises

The three exercises below correspond to the three core upgrades in this section: RMSNorm, SwiGLU, and GQA. Try to work through them yourself first before verifying with code.

On AI assistance: you can ask an AI to explain ideas and check direction, but please write the key lines of code yourself.


**Exercise 1: RMSNorm hand calculation**

Given the input vector $x = [3, 4]$ and scale parameter $\gamma = [1, 1]$, hand-compute the RMSNorm output.

RMSNorm formula:

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}$$

Requirements:
- First compute $\text{RMS}(x)$
- Then compute the two components of the output vector

Hint: $3^2 + 4^2 = 25$, so RMS is $\sqrt{25/2}$.


In [ ]:
# Exercise 1: RMSNorm hand calculation
import math

x = [3.0, 4.0]
gamma = [1.0, 1.0]

# TODO: compute RMS(x) = sqrt(mean(x_i^2))
rms = None

# TODO: compute RMSNorm output (each component = x_i / rms * gamma_i)
output = None

# Verify
assert rms is not None, 'Please replace the placeholder before running the assertion.'
assert output is not None, 'Please replace the placeholder before running the assertion.'

expected_rms = math.sqrt((9 + 16) / 2)
expected_output = [x[0] / expected_rms, x[1] / expected_rms]

assert abs(rms - expected_rms) < 0.01, f"RMS should be {expected_rms:.4f}, you got {rms:.4f}"
assert abs(output[0] - expected_output[0]) < 0.01
assert abs(output[1] - expected_output[1]) < 0.01

print(f"\✅ Exercise 1 passed")
print(f"   RMS = {rms:.4f}")
print(f"   output = [{output[0]:.4f}, {output[1]:.4f}]")
print(f"   Note: RMSNorm has no mean-subtraction step, computing one fewer statistic than LayerNorm.")


**Exercise 2: SwiGLU gating computation**

The SwiGLU formula is $\text{SwiGLU}(x) = (W_{up} \cdot x) \odot \text{Swish}(W_{gate} \cdot x)$.

Given a simplified 2D scenario: $x = [1, 2]$, and both $W_{up}$ and $W_{gate}$ are identity matrices (so $W_{up} \cdot x = x$ and $W_{gate} \cdot x = x$).

Requirements:
- Implement the Swish function: $\text{Swish}(a) = a \cdot \sigma(a) = \frac{a}{1 + e^{-a}}$
- Compute the SwiGLU output (information channel $\odot$ gating channel)

Hint: $\sigma(1) \approx 0.731$, $\sigma(2) \approx 0.881$. The gating channel's output lies in $(0, 1)$, effectively doing "proportional pass-through" on the information channel.


In [ ]:
# Exercise 2: SwiGLU gating computation
import math

x = [1.0, 2.0]

def sigmoid(a):
    return 1.0 / (1.0 + math.exp(-a))

def swish(a):
    # TODO: implement Swish(a) = a * sigmoid(a)
    return None

# TODO: compute SwiGLU output
up = x                  # W_up . x (W_up is the identity matrix)
gate = [swish(v) for v in x]   # first fill in the swish function
output = None           # element-wise multiply up (x) gate

# Verify
assert swish(1) is not None, "Please implement the swish function first"
assert output is not None, "Please compute output first"

expected_swish = [1.0 * sigmoid(1), 2.0 * sigmoid(2)]
expected_output = [1.0 * expected_swish[0], 2.0 * expected_swish[1]]

assert abs(swish(1) - expected_swish[0]) < 0.001
assert abs(swish(2) - expected_swish[1]) < 0.001
assert abs(output[0] - expected_output[0]) < 0.001
assert abs(output[1] - expected_output[1]) < 0.001

print(f"\✅ Exercise 2 passed")
print(f"   Swish([1, 2]) = [{swish(1):.4f}, {swish(2):.4f}]")
print(f"   SwiGLU output = [{output[0]:.4f}, {output[1]:.4f}]")
print(f"   Key observation: the gate values lie in (0, 1), doing 'proportional pass-through' on information,")
print(f"   rather than a hard cutoff like ReLU (either all pass or all delete).")


**Exercise 3: GQA KV Cache Capacity**

A model changes from MHA with 32 Q and 32 KV heads to GQA with 32 Q and 4 KV heads. Let `head_dim=128`, `seq_len=2048`, batch size 1, and FP16 values of 2 bytes.

Compute MHA cache bytes, GQA cache bytes, and the saving ratio.

Hint: cache per layer = $2	imes	ext{num\_kv\_heads}	imes	ext{head\_dim}	imes	ext{seq\_len}	imes	ext{batch}	imes	ext{bytes}$.


In [ ]:
# Exercise 3: GQA's KV Cache savings
num_q_heads = 32
num_kv_heads_mha = 32
num_kv_heads_gqa = 4
head_dim = 128
seq_len = 2048
batch_size = 1
bytes_per_element = 2  # FP16

# TODO: compute MHA's KV Cache size (in bytes)
# Formula: 2 * num_kv_heads * head_dim * seq_len * batch * bytes
mha_kv_bytes = None

# TODO: compute GQA's KV Cache size
gqa_kv_bytes = None

# TODO: compute GQA's savings ratio vs MHA
saving_ratio = None  # = 1 - gqa_kv_bytes / mha_kv_bytes

# Verify
assert mha_kv_bytes is not None, 'Please replace the placeholder before running the assertion.'
assert gqa_kv_bytes is not None, 'Please replace the placeholder before running the assertion.'
assert saving_ratio is not None, 'Please replace the placeholder before running the assertion.'

expected_mha = 2 * num_kv_heads_mha * head_dim * seq_len * batch_size * bytes_per_element
expected_gqa = 2 * num_kv_heads_gqa * head_dim * seq_len * batch_size * bytes_per_element
expected_saving = 1 - expected_gqa / expected_mha

assert mha_kv_bytes == expected_mha, f"MHA KV Cache should be {expected_mha} bytes"
assert gqa_kv_bytes == expected_gqa, f"GQA KV Cache should be {expected_gqa} bytes"
assert abs(saving_ratio - expected_saving) < 0.01, f"Savings ratio should be {expected_saving:.2%}"

print(f"✅ Exercise 3 passed")
print(f"   MHA KV Cache: {mha_kv_bytes / 1e6:.1f} MB")
print(f"   GQA KV Cache: {gqa_kv_bytes / 1e6:.1f} MB")
print(f"   Savings ratio: {saving_ratio:.1%}")
print(f"   Key observation: KV head count drops from 32 to 4, and KV Cache shrinks proportionally.")


## References

**Core components:** LLaMA; RoFormer (RoPE); RMSNorm; GLU Variants (SwiGLU).

**Attention evolution:** GQA; DeepSeek-V2 MLA; Native Sparse Attention.

**Representative combinations:** DeepSeek-V3 (MLA + MoE + MTP), Qwen3 (QK-Norm + RoPE), Kimi K2 (MuonClip + QK-Clip), GLM-4.5 (Muon + MTP), Gemma 3 (Sandwich Norm + Local/Global), and Llama 4 (iRoPE).

**Hybrid and linear Attention:** Jamba, MiniMax-01, DeepSeek V4, GLM-5.2, MiniMax M3, Qwen3.8, Kimi K3, and Nemotron 3 Ultra. See the links in the corresponding architecture sections above.
